# 02 — Random Forest: datasets y modelo base

Este notebook carga **Contact Lenses (discreto), Iris (continuo) y Statlog Heart
(mixto)**, prepara una partición estratificada reproducible y entrena un Random Forest
para cada uno. Las definiciones y fuentes completas están en el [README](../README.md).

**Contrato compartido:** `predict(X)` recibe un DataFrame con las características y
retorna un DataFrame con una única columna `prediccion`, conservando el índice y orden
de `X`. `X` e `y` se conservan como DataFrames; `y` tiene una columna `objetivo`.
El entrenamiento usa directamente scikit-learn, sin una clase propia.

Ejecutar todas las celdas en orden con el entorno del proyecto. Los datos de UCI están
incluidos en `data/raw/`; Iris viene con scikit-learn. La carga no utiliza Internet.

In [1]:
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.validation import check_is_fitted

SEMILLA = 42
FRACCION_TEST = 0.20

print(f"Python {platform.python_version()} | pandas {pd.__version__} | "
      f"NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

Python 3.12.8 | pandas 3.0.6 | NumPy 2.5.4 | scikit-learn 1.9.1


## 1. Carga y preparación de los datos

Los archivos originales se comprueban con SHA-256 para detectar cambios accidentales.
La preparación interpreta sus códigos según las descripciones originales, excluye
identificadores de las características y comprueba que no haya datos faltantes.
No se eliminan filas ni se imputan valores.

Contact Lenses tiene **cuatro** atributos: se incluye la producción de lágrimas,
según `lenses.names`. En Heart, las categorías codificadas con números se convierten
a texto para distinguirlas de las medidas numéricas.

In [2]:
def encontrar_raiz():
    """Permite ejecutar desde la raíz del repositorio o desde notebooks/."""
    for carpeta in (Path.cwd(), *Path.cwd().parents):
        if (carpeta / "data/raw/lenses.data").is_file():
            return carpeta
    raise FileNotFoundError(
        "No se encontró data/raw/lenses.data. Ejecutá desde el repositorio completo."
    )


RAIZ = encontrar_raiz()
HASHES = {
    "lenses.data": "711f22d5a4bed062bb715d39d23d54d2667206fbd97c94c7fd013ede47fb96c2",
    "heart.dat": "f5f3b4204c285bafadd85cb735f38b47689f2be7047feb172dcbeab648110bf9",
}


def leer_original(nombre, columnas):
    contenido = (RAIZ / "data/raw" / nombre).read_bytes()
    if sha256(contenido).hexdigest() != HASHES[nombre]:
        raise ValueError(f"El archivo {nombre} no coincide con la copia original.")
    return pd.read_csv(BytesIO(contenido), sep=r"\s+", header=None, names=columnas)


def cargar_datasets():
    """Devuelve {nombre: {X: DataFrame, y: DataFrame, tipo: str}}."""
    lenses = leer_original("lenses.data", [
        "id", "edad", "prescripcion", "astigmatismo", "lagrimas", "objetivo"
    ]).set_index("id")
    categorias_lenses = {
        "edad": {1: "joven", 2: "prepresbicia", 3: "presbicia"},
        "prescripcion": {1: "miopia", 2: "hipermetropia"},
        "astigmatismo": {1: "no", 2: "si"},
        "lagrimas": {1: "reducida", 2: "normal"},
    }
    X_lenses = lenses.drop(columns="objetivo").copy()
    for columna, etiquetas in categorias_lenses.items():
        X_lenses[columna] = X_lenses[columna].map(etiquetas)
    y_lenses = lenses[["objetivo"]].copy()
    y_lenses["objetivo"] = y_lenses["objetivo"].map(
        {1: "rigidos", 2: "blandos", 3: "sin_lentes"}
    )

    iris = load_iris(as_frame=True)
    X_iris = iris.data.copy()
    y_iris = iris.target.map(dict(enumerate(iris.target_names))).to_frame("objetivo")

    heart = leer_original("heart.dat", [
        "edad", "sexo", "dolor_pecho", "presion_reposo", "colesterol",
        "glucemia_alta", "ecg", "frecuencia_maxima", "angina_ejercicio",
        "depresion_st", "pendiente_st", "vasos", "thal", "objetivo"
    ])
    X_heart = heart.drop(columns="objetivo").copy()
    categoricas_heart = [
        "sexo", "dolor_pecho", "glucemia_alta", "ecg", "angina_ejercicio",
        "pendiente_st", "thal"
    ]
    for columna in categoricas_heart:
        X_heart[columna] = X_heart[columna].astype(int).astype(str)
    y_heart = heart[["objetivo"]].copy()
    y_heart["objetivo"] = y_heart["objetivo"].map({1: "ausencia", 2: "presencia"})

    return {
        "lenses": {"X": X_lenses, "y": y_lenses, "tipo": "discreto"},
        "iris": {"X": X_iris, "y": y_iris, "tipo": "continuo"},
        "heart": {"X": X_heart, "y": y_heart, "tipo": "mixto"},
    }


datasets = cargar_datasets()
TAMANOS = {"lenses": (24, 4), "iris": (150, 4), "heart": (270, 13)}
CLASES = {
    "lenses": {"rigidos": 4, "blandos": 5, "sin_lentes": 15},
    "iris": {"setosa": 50, "versicolor": 50, "virginica": 50},
    "heart": {"ausencia": 150, "presencia": 120},
}

resumen_datos = []
for nombre, datos in datasets.items():
    X, y = datos["X"], datos["y"]
    assert X.shape == TAMANOS[nombre]
    assert y.shape == (len(X), 1) and list(y.columns) == ["objetivo"]
    assert X.index.is_unique and X.index.equals(y.index)
    assert not X.isna().any().any() and not y.isna().any().any()
    assert np.isfinite(X.select_dtypes(include="number").to_numpy()).all()
    assert y["objetivo"].value_counts().to_dict() == CLASES[nombre]
    resumen_datos.append({
        "dataset": nombre, "tipo": datos["tipo"], "casos": len(X),
        "atributos": X.shape[1], "clases": y["objetivo"].nunique(),
        "numericos": X.select_dtypes(include="number").shape[1],
        "categoricos": X.select_dtypes(exclude="number").shape[1],
    })

display(pd.DataFrame(resumen_datos).set_index("dataset"))

,tipo,casos,atributos,clases,numericos,categoricos
dataset,,,,,,
lenses,discreto,24,4,3,0,4
iris,continuo,150,4,3,4,0
heart,mixto,270,13,2,6,7


## 2. Partición estratificada

Se reserva el 20 % para prueba, con semilla 42. `stratify` mantiene aproximadamente
las proporciones de las clases. Los índices originales se conservan para poder usar
**exactamente los mismos casos** en C4.5. La cantidad de filas debe ser entera: en
Contact Lenses quedan 19 casos de entrenamiento y 5 de prueba.

In [3]:
def particionar(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=FRACCION_TEST, random_state=SEMILLA,
        shuffle=True, stratify=y["objetivo"]
    )
    return dict(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)


particiones = {
    nombre: particionar(datos["X"], datos["y"])
    for nombre, datos in datasets.items()
}

distribuciones = []
for nombre, parte in particiones.items():
    X_train, X_test = parte["X_train"], parte["X_test"]
    y_train, y_test = parte["y_train"], parte["y_test"]
    assert X_train.index.intersection(X_test.index).empty
    assert len(X_train) + len(X_test) == len(datasets[nombre]["X"])
    assert X_train.index.equals(y_train.index)
    assert X_test.index.equals(y_test.index)
    assert set(y_train["objetivo"]) == set(y_test["objetivo"]) == set(CLASES[nombre])
    for clase, total in CLASES[nombre].items():
        distribuciones.append({
            "dataset": nombre, "clase": clase, "total": total,
            "train": int(y_train["objetivo"].eq(clase).sum()),
            "test": int(y_test["objetivo"].eq(clase).sum()),
        })

display(pd.DataFrame(distribuciones).set_index(["dataset", "clase"]))

total  train  test
dataset clase                         
lenses  rigidos         4      3     1
        blandos         5      4     1
        sin_lentes     15     12     3
iris    setosa         50     40    10
        versicolor     50     40    10
        virginica      50     40    10
heart   ausencia      150    120    30
        presencia     120     96    24

## 3. Modelo base y adaptación de `predict`

El pipeline transforma las categorías mediante one-hot y conserva las columnas
numéricas. El codificador aprende las categorías **solo en entrenamiento**. Una
categoría no observada en entrenamiento se representa con ceros en su grupo de
columnas (`handle_unknown="ignore"`). No se escalan las medidas numéricas.

El Random Forest tiene 100 árboles, Gini, bootstrap, `max_features="sqrt"`, sin límite
de profundidad y semilla 42. Se fija `n_jobs=1`. No hay búsqueda de hiperparámetros.

`crear_predictor` recibe el pipeline ya entrenado y devuelve una función `predict(X)`
asociada a ese modelo. La función verifica la entrada y convierte el array de
scikit-learn al DataFrame del contrato. No exige una clase nueva ni un entrenamiento
común con C4.5.

Referencias: [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
y [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [4]:
def crear_modelo(X_train):
    categoricas = X_train.select_dtypes(exclude="number").columns.tolist()
    numericas = X_train.select_dtypes(include="number").columns.tolist()
    preprocesador = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
        ("numericas", "passthrough", numericas),
    ])
    return Pipeline([
        ("preprocesador", preprocesador),
        ("random_forest", RandomForestClassifier(
            n_estimators=100, criterion="gini", max_depth=None,
            min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
            bootstrap=True, random_state=SEMILLA, n_jobs=1
        )),
    ])


def crear_predictor(modelo):
    """Asocia un pipeline entrenado a una función con el contrato predict(X)."""
    check_is_fitted(modelo.named_steps["random_forest"])
    columnas = pd.Index(modelo.feature_names_in_)

    def predict(X: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un pandas.DataFrame.")
        if not X.columns.equals(columnas):
            raise ValueError("X debe tener las mismas columnas y orden que en entrenamiento.")
        if X.empty or X.isna().any().any():
            raise ValueError("X debe contener casos y no tener valores faltantes.")
        return pd.DataFrame({"prediccion": modelo.predict(X)}, index=X.index.copy())

    return predict

## 4. Entrenar y predecir en los tres datasets

Se conserva un pipeline entrenado y una función de predicción por dataset.
`y_train` sigue siendo un DataFrame; se pasa su columna `objetivo` a scikit-learn,
que espera un vector de etiquetas para esta clasificación.

Las tablas muestran casos de prueba, sus respuestas reales y las predicciones.
La accuracy es la proporción de casos acertados y sirve como resultado base de esta
prueba; todavía no es una comparación con C4.5.

In [5]:
modelos = {}
predictores = {}
predicciones = {}
resultados = []

for nombre, parte in particiones.items():
    modelo = crear_modelo(parte["X_train"])
    modelo.fit(parte["X_train"], parte["y_train"]["objetivo"])
    predict = crear_predictor(modelo)
    salida = predict(parte["X_test"])

    modelos[nombre] = modelo
    predictores[nombre] = predict
    predicciones[nombre] = salida
    resultados.append({
        "dataset": nombre, "train": len(parte["X_train"]),
        "test": len(parte["X_test"]),
        "accuracy": accuracy_score(parte["y_test"]["objetivo"], salida["prediccion"]),
    })
    print(f"{nombre}: primeras predicciones de prueba")
    display(parte["y_test"].join(salida).head())

resultados = pd.DataFrame(resultados).set_index("dataset")
display(resultados)

lenses: primeras predicciones de prueba


,objetivo,prediccion
id,,
17,sin_lentes,sin_lentes
20,rigidos,sin_lentes
5,sin_lentes,sin_lentes
14,blandos,blandos
13,sin_lentes,sin_lentes


iris: primeras predicciones de prueba


,objetivo,prediccion
38,setosa,setosa
127,virginica,virginica
57,versicolor,versicolor
93,versicolor,versicolor
42,setosa,setosa


heart: primeras predicciones de prueba


,objetivo,prediccion
195,ausencia,ausencia
132,ausencia,ausencia
162,ausencia,ausencia
129,presencia,presencia
66,ausencia,ausencia


,train,test,accuracy
dataset,,,
lenses,19,5,0.800000
iris,120,30,0.900000
heart,216,54,0.796296


## 5. Verificación del contrato y de reproducibilidad

Se comprueba que la salida conserve índices, orden, cantidad de filas y etiquetas,
y que predecir no modifique la entrada. Se repiten la carga, la partición y el
entrenamiento para comprobar igualdad de datos y resultados con la misma semilla.
También se verifica que el codificador haya aprendido solo las categorías de train.

Estas comprobaciones detectan errores de integración. No exigen un umbral de accuracy:
con cinco casos de prueba en Contact Lenses, un solo error cambia el resultado en
20 puntos porcentuales. No se debe ajustar el modelo mirando este test.

In [6]:
datos_repetidos = cargar_datasets()
for nombre, parte in particiones.items():
    pd.testing.assert_frame_equal(datasets[nombre]["X"], datos_repetidos[nombre]["X"])
    pd.testing.assert_frame_equal(datasets[nombre]["y"], datos_repetidos[nombre]["y"])

    # Un orden invertido comprueba que predict no ordene ni reinicie los índices.
    entrada = parte["X_test"].iloc[::-1].copy(deep=True)
    original = entrada.copy(deep=True)
    salida = predictores[nombre](entrada)
    assert isinstance(salida, pd.DataFrame)
    assert salida.shape == (len(entrada), 1)
    assert list(salida.columns) == ["prediccion"]
    assert salida.index.equals(entrada.index)
    assert not salida.isna().any().any()
    assert set(salida["prediccion"]) <= set(CLASES[nombre])
    pd.testing.assert_frame_equal(entrada, original)
    pd.testing.assert_frame_equal(salida, predicciones[nombre].iloc[::-1])

    repetida = particionar(datos_repetidos[nombre]["X"], datos_repetidos[nombre]["y"])
    for clave in parte:
        pd.testing.assert_frame_equal(parte[clave], repetida[clave])

    categoricas = parte["X_train"].select_dtypes(exclude="number").columns
    if len(categoricas):
        encoder = modelos[nombre].named_steps["preprocesador"].named_transformers_["categoricas"]
        for columna, categorias in zip(categoricas, encoder.categories_):
            assert set(categorias) == set(parte["X_train"][columna])

    modelo_repetido = crear_modelo(repetida["X_train"])
    modelo_repetido.fit(repetida["X_train"], repetida["y_train"]["objetivo"])
    pd.testing.assert_frame_equal(
        predicciones[nombre], crear_predictor(modelo_repetido)(repetida["X_test"])
    )
    print(f"OK — {nombre}: carga, partición, contrato y predicciones reproducibles.")

OK — lenses: carga, partición, contrato y predicciones reproducibles.


OK — iris: carga, partición, contrato y predicciones reproducibles.


OK — heart: carga, partición, contrato y predicciones reproducibles.


## Uso del contrato para comparar después

Seleccionar el predictor entrenado del dataset y pasarle sus casos de prueba.
El mismo esquema de entrada/salida puede implementarse en C4.5. Comparar con
`y_test` usando los índices conservados; las etiquetas de respuesta nunca entran en `X`.

La comparación con C4.5 requiere su implementación y una ejecución sobre las mismas
particiones. Si solo procesa categorías, comenzar por Contact Lenses.

In [7]:
predict = predictores["iris"]
X_test = particiones["iris"]["X_test"]
y_test = particiones["iris"]["y_test"]
display(y_test.join(predict(X_test)).head())

,objetivo,prediccion
38,setosa,setosa
127,virginica,virginica
57,versicolor,versicolor
93,versicolor,versicolor
42,setosa,setosa
